# PDF Text Mining and WordCloud Analysis

This project extracts text from a PDF document, cleans and preprocesses the text, and creates a WordCloud that highlights the most frequent terms.

**Author:** Muhammad Kamil Shah


## Project objectives

- Extract text from a multi-page PDF.
- Normalize text by converting it to lowercase.
- Remove punctuation, stopwords, and non-alphabetic tokens.
- Save the cleaned text for reuse.
- Generate and export a WordCloud visualization.


In [ ]:
from pathlib import Path
import re
import string

import matplotlib.pyplot as plt
import nltk
import pdfplumber
from nltk.corpus import stopwords
from wordcloud import WordCloud

nltk.download("stopwords", quiet=True)


## 1. File paths

The notebook expects the repository structure shown in the README. Paths are resolved relative to the notebook location.


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_PATH = PROJECT_ROOT / "data" / "source_document.pdf"
CLEAN_TEXT_PATH = PROJECT_ROOT / "outputs" / "cleaned_text.txt"
WORDCLOUD_PATH = PROJECT_ROOT / "outputs" / "wordcloud_output.png"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")


## 2. Extract text from the PDF


In [ ]:
def extract_pdf_text(pdf_path: Path) -> str:
    """Extract text from every page of a PDF file."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            pages.append(text)
    return "\n".join(pages)

extracted_text = extract_pdf_text(PDF_PATH)
print(f"Extracted {len(extracted_text):,} characters.")
print(extracted_text[:500])


## 3. Clean and preprocess the text


In [ ]:
def clean_text(raw_text: str) -> str:
    """Normalize text and remove punctuation, numbers, and English stopwords."""
    stop_words = set(stopwords.words("english"))
    normalized = raw_text.lower()
    normalized = normalized.translate(str.maketrans("", "", string.punctuation))
    tokens = re.findall(r"[a-z]+", normalized)
    cleaned_tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    return " ".join(cleaned_tokens)

cleaned_text = clean_text(extracted_text)
print(f"Cleaned text contains {len(cleaned_text.split()):,} tokens.")
print(cleaned_text[:500])


## 4. Generate the WordCloud


In [ ]:
wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    collocations=False,
    max_words=200,
    random_state=42,
).generate(cleaned_text)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Most Frequent Terms in the Source Document")
plt.tight_layout()
plt.show()


## 5. Save the outputs


In [ ]:
CLEAN_TEXT_PATH.parent.mkdir(parents=True, exist_ok=True)
CLEAN_TEXT_PATH.write_text(cleaned_text, encoding="utf-8")
wordcloud.to_file(str(WORDCLOUD_PATH))

print(f"Saved cleaned text to: {CLEAN_TEXT_PATH}")
print(f"Saved WordCloud image to: {WORDCLOUD_PATH}")


## Conclusion

The workflow demonstrates a reproducible text-mining pipeline: PDF extraction, preprocessing, and visual exploration. The generated WordCloud provides a quick overview of the document's dominant terminology, while the cleaned text can support further tasks such as frequency analysis, topic modelling, or sentiment analysis.
